In [1]:
pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 6.8 MB/s  0:00:00
   ---------------------------------------- 0.0/802.4 kB ? eta -:--:--
   ---------------------------------------- 802.4/802.4 kB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/44.5 MB ? eta -:--:--
   -- ------------------------------------- 2.4/44.5 MB 12.5 MB/s eta 0:00:04
   ---- ----------------------------------- 5.0/44.5 MB 12.5 MB/s eta 0:00:04
   ------ --------------------------------- 7.6/44.5 MB 12.5 MB/s eta 0:00:03
   -------- ------------------------------- 9.2/44.5 MB 11.6 MB/s eta 0:00:04
   -------- ------------------------------- 10.0/44.5 MB 9.8 MB/s eta 0:00:04
   ---------- ----------------------------- 11.3/44.5 MB 9.3 MB/s eta 0:00:04
   ----------- ---------------------------- 13.1/44.5 MB 9.0 MB/s eta 0:00:04
   -------------

In [2]:
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO

# Configuration
frameWidth = 640
frameHeight = 480
cameraFeed = False  # Set to True to use camera feed
cameraNo = 1
videoPath = "robots.mp4"  # Specify your video file path
outputDir = "./output"  # Define output directory

# Load YOLOv5 model
try:
    model = YOLO("yolov5x.pt")  # This will automatically download 'yolov5x.pt' if not already downloaded
except Exception as e:
    raise RuntimeError(f"Failed to load YOLO model: {e}")

# Load class names from the model
classes = model.names
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# Create output directory if it doesn't exist
if not os.path.exists(outputDir):
    os.makedirs(outputDir)

# Initialize video capture
cap = cv2.VideoCapture(cameraNo if cameraFeed else videoPath)
if not cap.isOpened():
    raise ValueError(f"Cannot open video source: {videoPath if not cameraFeed else cameraNo}")

output_file = os.path.join(outputDir, os.path.basename(videoPath).replace('.mp4', '_Detection.avi')
                            if not cameraFeed else 'camera_feed_detection.avi')

# Get FPS and frame size
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0.0:
    fps = 25.0  # Default fallback FPS

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)

# Initialize VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Use a supported codec
video_writer = cv2.VideoWriter(output_file, fourcc, fps, frame_size)

if not video_writer.isOpened():
    raise ValueError(f"Failed to initialize VideoWriter with file: {output_file}")

starting_time = time.time()
frame_id = 0

# Processing loop
while True:
    success, img = cap.read()
    if not success:
        print('[i] ==> Done processing!!!')
        print('[i] ==> Output file is stored at', output_file)
        break

    frame_id += 1

    # Resize the frame for consistent processing
    resized_img = cv2.resize(img, (frameWidth, frameHeight))

    # Perform detection using YOLOv5x
    results = model(resized_img)  # Perform inference

    # Parse detections
    for result in results:  # Iterate through detections
        for box in result.boxes:
            x_min, y_min, x_max, y_max = map(int, box.xyxy[0].tolist())  # Get bounding box coordinates
            confidence = box.conf[0]  # Get confidence score
            class_id = int(box.cls[0])  # Get class ID

            if confidence > 0.65:  # Confidence threshold
                label = f"{classes[class_id]}: {confidence:.2f}%"
                color = colors[class_id]
                cv2.rectangle(resized_img, (x_min, y_min), (x_max, y_max), color, 2)
                cv2.putText(resized_img, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Display FPS
    elapsed_time = time.time() - starting_time
    fps = frame_id / elapsed_time
    cv2.putText(resized_img, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Write processed frame to video file
    video_writer.write(resized_img)

# Cleanup
cap.release()
video_writer.release()
cv2.destroyAllWindows()
print('==> All done!')

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\dai\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PRO TIP  Replace 'model=yolov5x.pt' with new 'model=yolov5xu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


0: 480x640 2 persons, 1 tv, 1492.5ms
Speed: 5.9ms preprocess, 1492.5ms inference, 29.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 chair, 1 tv, 879.3ms
Speed: 2.8ms preprocess, 879.3ms inference, 5.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 tv, 924.4ms
Speed: 3.2ms preprocess, 924.4ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0